In [1]:
from datetime import datetime
import pandas as pd
import networkx as nx
from visualize_ean_plotly import plot_ean_plotly, draw_ean_plotly, show_ean
from build_ean import build_ean, add_headway_arcs, propagate, enrich_trip_data_with_boundaries
import headway_integration as hi
import numpy as np
from collections import defaultdict
import sys
from pathlib import Path as path
import report as rp

project_root = path(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.schematic_map.routing import (build_route,get_signal_nodes_on_route,)
from infra_data.scenarios import load_network_csv, load_headways, get_scenario
from tools.RailML2trip_data.reassign_trip_id import reassign_trip_ids_by_departure
from tools.RailML2trip_data.add_side_nodes_to_trip_data import add_side_nodes_to_trip_data

In [6]:
from infra_data.scenarios import SCENARIOS

In [7]:
nodesDf = load_network_csv("nodes.csv", "node_id")
edgesDf = load_network_csv("edges.csv", "edge_id")
scenario = get_scenario()
scenario_dir = project_root / "results" / str(scenario)
scenario_dir.mkdir(parents=True, exist_ok=True)
trip_data = np.load(rf"C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\tools\RailML2trip_data\trip_data_{scenario}.npy", allow_pickle=True).item()
headway_dict = load_headways()
selected_trips_sorted = np.array(list(trip_data.keys()))

In [8]:
def _save_result(output_dir, key, value, scenario=None):
    if output_dir is None or scenario is None:
        return

    output_path = path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    results_file = output_path / f"{scenario}_results.npy"

    if results_file.exists():
        results = np.load(results_file, allow_pickle=True).item()
    else:
        results = {}

    results[key] = value
    np.save(results_file, results)


def compute_scenario_cost(edgesDf, nodesDf, scenario, output_dir):
    if scenario not in SCENARIOS:
        raise ValueError(f"Unknown scenario: {scenario}")

    scenario_sections = SCENARIOS[scenario]

    def is_in_scenario(value):
        return pd.notna(value) and value in scenario_sections

    def is_excluded(value):
        if isinstance(value, (set, list, tuple)):
            return bool(set(value) & scenario_sections)
        return pd.notna(value) and value in scenario_sections

    # Exclude existing/base infrastructure: costs apply only to new edges
    existing = edgesDf["scenario"].apply(
        lambda value: any(
            v.strip().lower() in {"base", "existing"}
            for v in str(value).split(",")
        ) if pd.notna(value) else False
    )
    
    included = edgesDf["scenario"].apply(is_in_scenario)
    excluded = edgesDf["exclude_scenario"].apply(is_excluded)

    scenario_edges = edgesDf[included & ~excluded & ~existing].copy()

    pk_rel = nodesDf["pk_rel"]

    node_from = scenario_edges["node_from"].map(pk_rel)
    node_to = scenario_edges["node_to"].map(pk_rel)

    cost = (node_to - node_from).abs().sum()

    _save_result(output_dir, "cost", cost, scenario=scenario)

    return cost, scenario_edges


cost, scenario_edges = compute_scenario_cost(
    edgesDf, nodesDf, scenario, output_dir=scenario_dir
)

cost

scenario_edges.to_csv(
    scenario_dir / f"scenario_edges_{scenario}.csv",
    index=False
)

In [9]:
scenarios = ["1a", "1b", "1c", "3a", "3b", "3c", "3d", "3e", "3ee", "3f", "4"]

for scenario in scenarios:
    scenario_dir = project_root / "results" / str(scenario)
    scenario_dir.mkdir(parents=True, exist_ok=True)
    cost, scenario_edges = compute_scenario_cost(edgesDf, nodesDf, scenario, output_dir=scenario_dir)
    cost
    scenario_edges.to_csv(scenario_dir / f"scenario_edges_{scenario}.csv", index=False)

In [10]:
cost

np.float64(11.192999999999998)

In [ ]:
scenario = "1b"

scenario_dir = project_root / "results" / str(scenario)
scenario_dir.mkdir(parents=True, exist_ok=True)
cost, scenario_edges = compute_scenario_cost(edgesDf, nodesDf, scenario, output_dir=scenario_dir)
scenario_edges.to_csv(scenario_dir / f"scenario_edges_{scenario}.csv", index=False)
cost

np.float64(12.889)